# Proposition Graph Construction (PropRAG-style, LLM-free clique/containment + LLM-resolved synonymy)

Builds the three edge types from the PropRAG paper (Section 5.1 / Appendix A.1) directly from an
already-computed OpenIE output file — **without** importing or running `PropRAG.py` /
`graph_beam_search.py`:

1. **Entity Clique (hyper-edge) edges** — deterministic, built from proposition co-occurrence.
2. **Passage Containment edges** — deterministic, built from passage→entity membership.
3. **Synonymy edges** — the one genuinely fuzzy step, resolved with a local Gemma E4B model over
   candidate blocks (entities sharing a significant word), since checking all ~13k entities pairwise
   is infeasible.

Scope: gold supporting-fact passages for **bridge-type** HotpotQA questions only (1,618 passages /
8,055 propositions / 13,460 unique entities), joined by passage title between
`reproduce/dataset/hotpotqa.json` and the OpenIE output.

In [ ]:
import threading
from typing import List, Optional

import torch
from transformers import AutoModelForCausalLM, AutoProcessor, BitsAndBytesConfig


class GemmaLLM:
    def __init__(
        self,
        model_path: str,
        device_map=None,
        load_in_4bit: bool = True,
        default_max_new_tokens: int = 1024,
        default_temperature: float = 0.2,
        default_top_p: float = 0.9,
    ):
        self.model_path = model_path
        self.default_max_new_tokens = default_max_new_tokens
        self.default_temperature = default_temperature
        self.default_top_p = default_top_p

        # A lock because the pipeline may call generate() from multiple
        # logical "workers" (e.g. when batching community reports) - a single
        # local GPU model is not safely reentrant, so we serialize calls.
        self._lock = threading.Lock()

        self.processor = AutoProcessor.from_pretrained(model_path)
        self.processor.tokenizer.padding_side = "left"
        if self.processor.tokenizer.pad_token is None:
            self.processor.tokenizer.pad_token = self.processor.tokenizer.eos_token

        quant_config = None
        if load_in_4bit:
            quant_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True,
            )

        self.model = AutoModelForCausalLM.from_pretrained(
            model_path,
            quantization_config=quant_config,
            dtype="auto",
            low_cpu_mem_usage=True,
            device_map=device_map or {"": 0},
            attn_implementation="sdpa",
        )
        self.model.eval()

    def _build_gen_kwargs(self, max_new_tokens, temperature, top_p, pad_token_id):
        # IMPORTANT: temperature=0.0 means "greedy decoding", which is
        # different from "use the default temperature". Treat None (not
        # given) as "use the default"; treat 0.0 as an explicit request for
        # greedy decoding, and don't pass a temperature/top_p to
        # model.generate() at all in that case - some transformers versions
        # validate temperature even when do_sample=False and will raise a
        # ValueError on temperature=0.0.
        effective_temperature = temperature if temperature is not None else self.default_temperature
        do_sample = effective_temperature > 0

        gen_kwargs = dict(
            max_new_tokens=max_new_tokens or self.default_max_new_tokens,
            do_sample=do_sample,
            pad_token_id=pad_token_id,
        )
        if do_sample:
            gen_kwargs["temperature"] = effective_temperature
            gen_kwargs["top_p"] = top_p if top_p is not None else self.default_top_p
        return gen_kwargs

    def generate(
        self,
        prompt: str,
        system: Optional[str] = None,
        max_new_tokens: Optional[int] = None,
        temperature: Optional[float] = None,
        top_p: Optional[float] = None,
    ) -> str:
        """Runs one chat-formatted generation and returns only the newly
        generated text (the prompt itself is stripped out)."""
        messages = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": prompt})

        inputs = self.processor.tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
        ).to(self.model.device)

        gen_kwargs = self._build_gen_kwargs(
            max_new_tokens, temperature, top_p, self.processor.tokenizer.pad_token_id
        )

        with self._lock:
            with torch.no_grad():
                output_ids = self.model.generate(**inputs, **gen_kwargs)

        input_len = inputs["input_ids"].shape[1]
        new_tokens = output_ids[:, input_len:]
        text = self.processor.tokenizer.decode(new_tokens[0], skip_special_tokens=True)
        return text.strip()

    def generate_batch(
        self,
        prompts: List[str],
        system: Optional[str] = None,
        max_new_tokens: Optional[int] = None,
        temperature: Optional[float] = None,
        top_p: Optional[float] = None,
    ) -> List[str]:
        """Runs one batched chat-formatted generation over multiple independent prompts and
        returns the newly generated text for each, in the same order as `prompts`.

        Relies on left-padding (set in __init__) so every row's real input ends at the same
        tensor position - that lets a single `output_ids[:, input_len:]` slice correctly
        extract just the new tokens for every row at once, same as the single-prompt path.
        """
        tokenizer = self.processor.tokenizer
        texts = []
        for prompt in prompts:
            messages = []
            if system:
                messages.append({"role": "system", "content": system})
            messages.append({"role": "user", "content": prompt})
            texts.append(tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False))

        inputs = tokenizer(
            texts, return_tensors="pt", padding=True, add_special_tokens=False
        ).to(self.model.device)

        gen_kwargs = self._build_gen_kwargs(max_new_tokens, temperature, top_p, tokenizer.pad_token_id)

        with self._lock:
            with torch.no_grad():
                output_ids = self.model.generate(**inputs, **gen_kwargs)

        input_len = inputs["input_ids"].shape[1]
        new_tokens = output_ids[:, input_len:]
        decoded = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
        return [t.strip() for t in decoded]

In [ ]:
import json
import re
import pickle
import itertools
import collections
from pathlib import Path

import networkx as nx
from tqdm.auto import tqdm

# --- Config ---------------------------------------------------------------
OPENIE_PATH = "PropRAG/outputs/hotpotqa/openie_results_ner_meta-llama_llama-3.3-70b-instruct.json"
QUERIES_PATH = "PropRAG/reproduce/dataset/hotpotqa.json"

# TODO: point this at your local Gemma E4B checkpoint (local path or HF hub id).
GEMMA_MODEL_PATH = "<path-or-HF-id-of-your-Gemma-E4B-checkpoint>"

QUESTION_TYPE = "bridge"          # which HotpotQA question type to scope the graph to

SYNONYMY_BLOCK_SIZE_CAP = 20      # max entities per LLM synonymy-resolution call; larger token
                                   # posting lists are split into consecutive chunks of this size
SYNONYMY_BATCH_SIZE = 8           # how many blocks to run per generate() call; tune down if you OOM
                                   # on a 12GB GPU, tune up if you have headroom to spare
LLM_DRY_RUN_LIMIT = 10            # set to None for a full run; small int for a quick sanity check first

OUTPUT_DIR = Path("bridge_gold_graph")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
with open(QUERIES_PATH) as f:
    queries = json.load(f)

bridge_gold_titles = {
    title
    for item in queries
    if item.get("type") == QUESTION_TYPE
    for title, _ in item["supporting_facts"]
}

print(f"{len(queries)} total queries")
print(f"{sum(1 for q in queries if q.get('type') == QUESTION_TYPE)} queries of type '{QUESTION_TYPE}'")
print(f"{len(bridge_gold_titles)} unique gold supporting-fact titles for '{QUESTION_TYPE}' questions")

In [ ]:
with open(OPENIE_PATH) as f:
    openie_docs = json.load(f)["docs"]


def doc_title(doc):
    # Each passage is formatted as "<Title>\n<body...>" by the OpenIE indexing step.
    return doc["passage"].split("\n", 1)[0]


docs = [d for d in openie_docs if doc_title(d) in bridge_gold_titles]

num_propositions = sum(len(d["propositions"]) for d in docs)
unique_entities = {e for d in docs for p in d["propositions"] for e in p["entities"]}

print(f"{len(openie_docs)} total passages in the OpenIE corpus")
print(f"{len(docs)} passages in scope (expected 1618)")
print(f"{num_propositions} propositions in scope (expected 8055)")
print(f"{len(unique_entities)} unique entities in scope (expected 13460)")

## 1. Deterministic edges: Entity Clique (hyper-edge) and Passage Containment

Both are exact facts already present in the OpenIE output — no LLM judgment needed. Mirrors
`PropRAG.py`'s `add_proposition_edges_with_entity_connections` (pairwise clique per proposition,
weight accumulates across propositions) and `add_passage_edges` (passage → every entity that appears
in one of its propositions, weight 1.0).

In [ ]:
clique_weights = collections.defaultdict(float)       # (entity, entity) -> weight, entities sorted so (e1, e2) is unique per pair
containment_weights = collections.defaultdict(float)   # (passage_id, entity) -> weight
passage_text = {}

for doc in docs:
    pid = doc["idx"]
    passage_text[pid] = doc["passage"]

    doc_entities = set()
    for prop in doc["propositions"]:
        prop_entities = sorted(set(prop["entities"]))
        doc_entities.update(prop_entities)
        for e1, e2 in itertools.combinations(prop_entities, 2):
            clique_weights[(e1, e2)] += 1.0

    for e in doc_entities:
        containment_weights[(pid, e)] = 1.0

all_entities = sorted(unique_entities)

print(f"{len(passage_text)} passage nodes")
print(f"{len(all_entities)} entity nodes")
print(f"{len(clique_weights)} clique (hyper-edge) entity-entity pairs")
print(f"{len(containment_weights)} passage-containment edges")

## 2. Synonymy edges — candidate blocking

Checking all ~13k entities pairwise (~90M pairs) is infeasible for an LLM. Instead, build an inverted
index from significant words to the entities that contain them, and treat entities that share a
significant word as a candidate block worth checking together. Blocks larger than
`SYNONYMY_BLOCK_SIZE_CAP` (e.g. a generic token like "american" or "film") are split into consecutive
chunks of that size rather than dropped, so recall isn't sacrificed entirely — cross-chunk matches
within one oversized token bucket are the one known gap, called out in the notebook's limitations note
below.

Each candidate entity is paired with one example proposition sentence it appeared in, used as
disambiguating context in the LLM prompt.

In [ ]:
STOPWORDS = {
    "a", "an", "the", "of", "in", "on", "at", "and", "or", "is", "was", "were",
    "to", "for", "by", "with", "from",
}


def significant_tokens(entity):
    toks = re.sub(r"[^a-z0-9\s]", " ", entity.lower()).split()
    sig = [t for t in toks if t not in STOPWORDS and len(t) >= 3 and not t.isdigit()]
    return sig or [entity.lower().strip()]  # entity has no significant token -> block on itself only


# one example proposition sentence per entity, for disambiguation context in the prompt
entity_example_context = {}
for doc in docs:
    for prop in doc["propositions"]:
        for e in prop["entities"]:
            entity_example_context.setdefault(e, prop["text"])

inverted = collections.defaultdict(set)
for ent in all_entities:
    for tok in significant_tokens(ent):
        inverted[tok].add(ent)

blocks = []
seen_block_keys = set()
for tok, ents in inverted.items():
    ents = sorted(ents)
    if len(ents) < 2:
        continue  # nothing to resolve for this token
    chunks = [ents[i:i + SYNONYMY_BLOCK_SIZE_CAP] for i in range(0, len(ents), SYNONYMY_BLOCK_SIZE_CAP)]
    for chunk in chunks:
        if len(chunk) < 2:
            continue
        key = tuple(chunk)
        if key not in seen_block_keys:
            seen_block_keys.add(key)
            blocks.append(chunk)

print(f"{len(inverted)} distinct blocking tokens")
print(f"{len(blocks)} candidate synonym blocks (max {SYNONYMY_BLOCK_SIZE_CAP} entities each)")

## 3. Synonymy resolution prompt

Given one block of candidate entities (with disambiguating context), ask the LLM to cluster the ones
that refer to the same real-world entity. Every input entity must land in exactly one cluster;
entities with no synonym in the block form a singleton cluster.

In [ ]:
SYNONYMY_PROMPT_TEMPLATE = '''Your task is to identify which of the following entity mentions, extracted from a text corpus, refer \
to the exact same real-world entity or concept (person, place, organization, work, date, etc.), \
accounting for aliases, abbreviations, partial names, and different surface forms.

Group the entities into clusters. Two entities belong in the same cluster ONLY if they refer to the \
identical real-world thing. Do not merge entities that are merely related, in the same category, or \
share a word but denote different things (e.g. "Saint Peter" and "Saint Paul" are different people; \
"University of Chicago" and "University of Illinois" are different universities).

Each entity below is shown with one example sentence it appeared in, to help disambiguate. Every \
input entity must appear in exactly one cluster. Entities with no synonym in this list form a \
singleton cluster of size 1.

Respond with JSON only, in this exact format — each cluster must be a JSON array using square \
brackets [ ], not curly braces:
{{"clusters": [["entity a", "entity b"], ["entity c"]]}}

Demonstration:
Input entities:
1. "Saint Peter" — context: "Mantua Cathedral is a Roman Catholic cathedral dedicated to Saint Peter."
2. "St. Peter's Basilica" — context: "St. Peter's Basilica is located in Vatican City."
3. "St. Peter" — context: "The Basilica of St. Peter is the reputed burial place of St. Peter."
4. "Saint Paul" — context: "The cathedral is also dedicated to Saint Paul."

Output:
{{"clusters": [["Saint Peter", "St. Peter"], ["St. Peter's Basilica"], ["Saint Paul"]]}}

Input entities:
{entity_list}
'''


def build_synonymy_prompt(block, context_map):
    lines = [
        f'{i + 1}. "{ent}" — context: "{context_map.get(ent, "")}"'
        for i, ent in enumerate(block)
    ]
    return SYNONYMY_PROMPT_TEMPLATE.format(entity_list="\n".join(lines))


_JSON_OBJECT_RE = re.compile(r"\{.*\}", re.DOTALL)


def _fix_set_literals(text):
    """Repair Python-style set literals like {"a", "b"} into JSON arrays ["a", "b"],
    anywhere they appear -- including nested inside a {"clusters": [...]} wrapper.
    Recurses into inner content first, then converts the current level only if
    it has no ':' (i.e. it's a bare set of values, not a JSON object)."""
    out = []
    i = 0
    n = len(text)
    while i < n:
        ch = text[i]
        if ch != "{":
            out.append(ch)
            i += 1
            continue

        depth = 1
        j = i + 1
        while j < n and depth > 0:
            if text[j] == "{":
                depth += 1
            elif text[j] == "}":
                depth -= 1
            j += 1

        if depth != 0:
            out.append(text[i:])
            break

        inner = text[i + 1:j - 1]
        fixed_inner = _fix_set_literals(inner)  # recurse first

        if ":" not in inner:
            out.append("[" + fixed_inner + "]")   # bare set -> JSON array
        else:
            out.append("{" + fixed_inner + "}")   # real object -> keep braces

        i = j

    return "".join(out)


def parse_clusters(raw_text, block):
    """Extract {"clusters": [[...], ...]} from raw LLM output, validated against the input block.
    Returns None on any parse failure (caller logs the failure and moves on)."""
    if raw_text is None:
        return None

    match = _JSON_OBJECT_RE.search(raw_text)
    if not match:
        return None

    candidate = match.group(0)
    try:
        parsed = json.loads(candidate)
    except json.JSONDecodeError:
        try:
            parsed = json.loads(_fix_set_literals(candidate))
        except json.JSONDecodeError:
            return None

    try:
        clusters = parsed["clusters"]
    except (KeyError, TypeError):
        return None

    block_set = set(block)
    cleaned = []
    for cluster in clusters:
        if not isinstance(cluster, list):
            continue
        cluster = sorted({e for e in cluster if e in block_set})
        if cluster:
            cleaned.append(cluster)
    return cleaned

## 4. Run synonymy resolution over all candidate blocks

Loads the local Gemma E4B model (see `GemmaLLM` above) and runs generation over blocks in batches of
`SYNONYMY_BATCH_SIZE` (via `generate_batch`) rather than one at a time, since decoding a small model
at batch size 1 leaves most of the GPU idle. Progress is checkpointed to `synonymy_audit.jsonl` /
`failed_blocks.json` every 25 batches, on every OOM recovery, and at the end — and **re-running this
cell resumes from that checkpoint** (blocks already present in `synonymy_audit.jsonl` are skipped), so
stopping the kernel or hitting a crash doesn't mean starting over. Start with `LLM_DRY_RUN_LIMIT` set
small (e.g. 20-30) to sanity-check the model, prompt, and parser — and to get a measured per-batch
timing / full-run ETA — before committing to a full run over all blocks. If you hit a CUDA OOM it's
handled automatically (batch is halved and retried); if that happens often, lower
`SYNONYMY_BATCH_SIZE` before continuing.

In [ ]:
import time

llm = GemmaLLM(GEMMA_MODEL_PATH)

synonymy_weights = collections.defaultdict(float)  # (entity, entity) -> weight
failed_blocks = []
audit_log = []
num_oom_retries = 0

AUDIT_LOG_PATH = OUTPUT_DIR / "synonymy_audit.jsonl"
FAILED_BLOCKS_PATH = OUTPUT_DIR / "failed_blocks.json"


def save_checkpoint():
    with open(AUDIT_LOG_PATH, "w") as f:
        for entry in audit_log:
            f.write(json.dumps(entry) + "\n")
    with open(FAILED_BLOCKS_PATH, "w") as f:
        json.dump(failed_blocks, f, indent=2)


# --- Resume from a previous run, if a checkpoint exists ------------------
# Keyed on the exact block contents, so changing SYNONYMY_BLOCK_SIZE_CAP or the blocking
# logic between runs correctly invalidates the old checkpoint instead of silently misapplying it.
already_done = set()
if AUDIT_LOG_PATH.exists():
    with open(AUDIT_LOG_PATH) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            entry = json.loads(line)
            audit_log.append(entry)
            already_done.add(tuple(entry["block"]))
            for cluster in entry["clusters"]:
                for e1, e2 in itertools.combinations(sorted(set(cluster)), 2):
                    synonymy_weights[(e1, e2)] += 1.0
    if FAILED_BLOCKS_PATH.exists():
        with open(FAILED_BLOCKS_PATH) as f:
            failed_blocks = json.load(f)
    print(f"Resuming: {len(already_done)} blocks already completed in a previous run "
          f"({AUDIT_LOG_PATH})")


def safe_generate_batch(block_batch):
    """Generate for a batch of blocks; on CUDA OOM, clear the cache and retry as two
    half-size batches (recursively, down to batch size 1) instead of crashing the run.
    Returns one raw output string per block in block_batch order - a block that still OOMs
    even alone gets None, which the caller treats like any other unparseable response."""
    global num_oom_retries
    prompts = [build_synonymy_prompt(b, entity_example_context) for b in block_batch]
    try:
        return llm.generate_batch(prompts, temperature=0.0)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        num_oom_retries += 1
        if len(block_batch) == 1:
            print(f"  OOM on a single block even after retry - skipping: {block_batch[0][0]!r}")
            return [None]
        mid = len(block_batch) // 2
        print(f"  OOM on batch of {len(block_batch)} - retrying as {mid} + {len(block_batch) - mid}")
        save_checkpoint()  # OOM is exactly the moment we don't want to lose progress
        return safe_generate_batch(block_batch[:mid]) + safe_generate_batch(block_batch[mid:])


run_blocks = blocks if LLM_DRY_RUN_LIMIT is None else blocks[:LLM_DRY_RUN_LIMIT]
run_blocks = [b for b in run_blocks if tuple(b) not in already_done]
block_batches = [run_blocks[i:i + SYNONYMY_BATCH_SIZE] for i in range(0, len(run_blocks), SYNONYMY_BATCH_SIZE)]
call_times = []  # per-batch wall time

for batch_i, block_batch in enumerate(tqdm(block_batches)):
    t0 = time.perf_counter()
    raw_outputs = safe_generate_batch(block_batch)
    call_times.append(time.perf_counter() - t0)

    for block, raw in zip(block_batch, raw_outputs):
        clusters = parse_clusters(raw, block) if raw is not None else None

        if clusters is None:
            failed_blocks.append({"block": block, "raw": raw})
            clusters = []

        audit_log.append({"block": block, "raw": raw, "clusters": clusters})

        for cluster in clusters:
            for e1, e2 in itertools.combinations(sorted(set(cluster)), 2):
                synonymy_weights[(e1, e2)] += 1.0

    if batch_i % 25 == 0:
        save_checkpoint()

save_checkpoint()

print(f"Ran {len(run_blocks)} new blocks this session "
      f"({len(audit_log)} / {len(blocks)} total completed across all sessions)")
print(f"{num_oom_retries} OOM retries occurred (batch auto-split and resumed each time)")
print(f"{len(failed_blocks)} blocks failed to parse (includes any unrecoverable single-block OOMs)")
print(f"{len(synonymy_weights)} synonymy entity-entity pairs discovered")

# Timing / ETA for the full run, measured on whatever subset just ran this session.
if call_times:
    avg_batch_s = sum(call_times) / len(call_times)
    avg_block_s = avg_batch_s / SYNONYMY_BATCH_SIZE
    print(f"\navg {avg_batch_s:.2f}s/batch ({avg_block_s:.2f}s/block equivalent), "
          f"range {min(call_times):.2f}-{max(call_times):.2f}s/batch")
    remaining = len(blocks) - len(audit_log)
    if remaining > 0:
        remaining_batches = -(-remaining // SYNONYMY_BATCH_SIZE)  # ceil division
        eta_s = avg_batch_s * remaining_batches
        print(f"estimated time for the {remaining} still-remaining blocks: {eta_s / 3600:.1f} hours "
              f"[measured on only {len(block_batches)} batches this session -- treat as rough]")
if num_oom_retries:
    print(f"\n{num_oom_retries} OOM(s) occurred during this run - consider lowering SYNONYMY_BATCH_SIZE "
          f"(currently {SYNONYMY_BATCH_SIZE}) before continuing, to avoid the retry overhead.")

## 5. Assemble the proposition graph

Entity nodes and passage nodes, with all three edge types layered in. Entity-entity pairs that are
linked by *both* a clique co-occurrence and a synonymy relationship keep both weights as separate edge
attributes rather than overwriting each other.

In [ ]:
def entity_node(ent):
    return f"entity::{ent}"


G = nx.Graph()

for pid, text in passage_text.items():
    G.add_node(pid, node_type="passage", text=text)

for ent in all_entities:
    G.add_node(entity_node(ent), node_type="entity", text=ent)


def add_entity_edge(e1, e2, kind, weight):
    u, v = entity_node(e1), entity_node(e2)
    if G.has_edge(u, v):
        G[u][v][f"{kind}_weight"] = G[u][v].get(f"{kind}_weight", 0.0) + weight
    else:
        G.add_edge(u, v, **{f"{kind}_weight": weight})


for (e1, e2), w in clique_weights.items():
    add_entity_edge(e1, e2, "hyperedge", w)

for (e1, e2), w in synonymy_weights.items():
    add_entity_edge(e1, e2, "synonymy", w)

for (pid, ent), w in containment_weights.items():
    G.add_edge(pid, entity_node(ent), containment_weight=w)

print(f"{G.number_of_nodes()} total nodes, {G.number_of_edges()} total edges")

## 6. Save outputs

In [ ]:
with open(OUTPUT_DIR / "graph.gpickle", "wb") as f:
    pickle.dump(G, f)

graphml_copy = G.copy()
for _, data in graphml_copy.nodes(data=True):
    for k, v in list(data.items()):
        data[k] = str(v)
for _, _, data in graphml_copy.edges(data=True):
    for k, v in list(data.items()):
        data[k] = str(v)
nx.write_graphml(graphml_copy, OUTPUT_DIR / "graph.graphml")

save_checkpoint()  # final synonymy_audit.jsonl / failed_blocks.json write

print(f"Saved graph.gpickle, graph.graphml, synonymy_audit.jsonl, failed_blocks.json to {OUTPUT_DIR}/")

## 7. Summary stats

In [ ]:
num_passage_nodes = sum(1 for _, d in G.nodes(data=True) if d["node_type"] == "passage")
num_entity_nodes = sum(1 for _, d in G.nodes(data=True) if d["node_type"] == "entity")

num_hyperedge = sum(1 for _, _, d in G.edges(data=True) if "hyperedge_weight" in d)
num_synonymy = sum(1 for _, _, d in G.edges(data=True) if "synonymy_weight" in d)
num_containment = sum(1 for _, _, d in G.edges(data=True) if "containment_weight" in d)

singleton_clusters = sum(1 for e in audit_log for c in e["clusters"] if len(c) == 1)
merged_clusters = sum(1 for e in audit_log for c in e["clusters"] if len(c) > 1)

print("=== Graph summary ===")
print(f"passage nodes:     {num_passage_nodes}")
print(f"entity nodes:      {num_entity_nodes}")
print(f"hyperedge edges:   {num_hyperedge}")
print(f"containment edges: {num_containment}")
print(f"synonymy edges:    {num_synonymy}")
print()
print("=== Synonymy resolution ===")
print(f"blocks run:            {len(audit_log)}")
print(f"blocks failed to parse:{len(failed_blocks)}")
print(f"merged (size>1) clusters found: {merged_clusters}")
print(f"singleton clusters found:       {singleton_clusters}")
print()
print("=== Example merged synonymy clusters ===")
shown = 0
for entry in audit_log:
    for cluster in entry["clusters"]:
        if len(cluster) > 1:
            print(" ", cluster)
            shown += 1
        if shown >= 10:
            break
    if shown >= 10:
        break